# Core Functions – Metadata Driven JSON Loader

## 1. Small helper

In [0]:
def get_full_table_name(catalog, schema, table):
    return f"{catalog}.{schema}.{table}"

## 2. source_control table

The table you maintain by hand one row per source folder. Flip
`is_active` to 0 to pause a source without deleting it.

| column | meaning |
|---|---|
| s_no | serial number |
| source | s3/adls/volume/dbfs |
| file_path | folder to scan (must end with `/`) |
| file_type | json |
| tgt_catalog / tgt_schema / tgt_table | where files from this source get loaded - all files under one source share one target table |
| tgt_checkpoint_path | base checkpoint/schema location for Autoloader |
| is_full_load | 1 = `file_path` is a folder of date subfolders (e.g. `20260901/`, `20260902/`); every run re-scans ALL of them. Each subfolder gets its own Autoloader checkpoint/schema (`tgt_checkpoint_path_<folder>`), so re-scanning a subfolder that's already fully loaded is cheap - Autoloader's own checkpoint sees no new files and exits almost immediately. |
| is_incremental_load | 1 = `file_path` is a folder of date subfolders, but only today's folder (system date, `yyyyMMdd`) is processed each run. If today's folder doesn't exist yet, that source is skipped for the run with a log entry. |
| is_active | 1 = pipeline processes this source, 0 = skipped |

In [0]:
def get_active_sources():
    ctl_full = get_full_table_name(meta_catalog, meta_schema, source_control_table)
    df = spark.table(ctl_full).filter(F.col("is_active") == 1)
    return [row.asDict() for row in df.collect()]

## 3. load_logs table

There's no separate file-status table. The log table itself is checked on
every run to work out which files already loaded successfully — a file's
**most recent** SUCCESS or FAILED log row is its current status.

| column | meaning |
|---|---|
| log_id | unique id for the log row |
| run_id | run id |
| executed_by | executor name |
| s_no | serial number |
| source / file_path / tgt_catalog / tgt_schema / tgt_table | target delta path|
| file_name | file name |
| status | SUCCESS / FAILED |
| start_time / end_time | timing |
| rows_processed | row count|
| message | detail, including error messages |

In [0]:
def write_log(status, source_row=None, file_name=None, rows_processed=None, message=None, start_time=None):

    now = datetime.now()
    sr = source_row or {}
    row = [(
        str(uuid.uuid4()), run_id, executed_by,
        sr.get("s_no"), sr.get("source"), sr.get("file_path"),
        sr.get("tgt_catalog"), sr.get("tgt_schema"), sr.get("tgt_table"),
        file_name, status, start_time or now, now, rows_processed, message,
    )]
    df_log = spark.createDataFrame(row, schema=LOG_SCHEMA)
    df_log.write.format("delta").mode("append") \
        .saveAsTable(get_full_table_name(meta_catalog, meta_schema, log_table))

## 4. File discovery and deciding what to process

`discover_files` lists every file currently sitting in a source's folder.

`get_files_to_process` then looks at the log table for that source, finds
each file's most recent SUCCESS/FAILED result, and returns:
- files that were never logged before (brand new files), plus
- files whose last attempt was FAILED

Files whose last attempt was SUCCESS are left out, so a re-run only
touches what's new or previously failed.

In [0]:
def discover_files(source_row):
    files = dbutils.fs.ls(source_row["file_path"])
    ext = "." + source_row["file_type"]
    return [f.name for f in files if f.name.endswith(ext)]


def get_latest_file_status(s_no):
    log_full = get_full_table_name(meta_catalog, meta_schema, log_table)
    df = (
        spark.table(log_full)
             .filter(F.col("s_no") == s_no)
             .filter(F.col("status").isin("SUCCESS", "FAILED"))
             .filter(F.col("file_name").isNotNull())
    )
    window = Window.partitionBy("file_name").orderBy(desc("end_time"))
    latest = (
        df.withColumn("rn", row_number().over(window))
          .filter(F.col("rn") == 1)
          .select("file_name", "status")
    )
    return {row.file_name: row.status for row in latest.collect()}


def get_files_to_process(source_row, discovered_files):
    latest_status = get_latest_file_status(source_row["s_no"])
    return [f for f in discovered_files if latest_status.get(f) != "SUCCESS"]

## 4b. Date-subfolder discovery (for is_full_load / is_incremental_load)

`list_date_subfolders` lists the immediate subfolders of a source's
`file_path` and keeps only the ones that look like a date, `yyyyMMdd`
(e.g. `20260901`). Non-matching entries (other folders, stray files)
are ignored. Returned in sorted (chronological) order.

In [0]:
_DATE_FOLDER_PATTERN = re.compile(r"^\d{8}$")

def list_date_subfolders(base_path):
    entries = dbutils.fs.ls(base_path)
    folders = [
        e.name.rstrip("/")
        for e in entries
        if e.name.endswith("/") and _DATE_FOLDER_PATTERN.match(e.name.rstrip("/"))
    ]
    return sorted(folders)

In [0]:
def extract_dt_data_file(file_name):
    stem = file_name.rsplit(".", 1)[0]  # drop extension, if any
    parts = stem.split("_")
    return parts[-2] if len(parts) >= 2 else None

## 5. Autoloader ingestion

`process_file` / `run_pipeline` above load one file per Spark job — fine
for a handful of files, far too slow once a source has tens of thousands
(each file pays for a separate text-read job, a `.count()` job, and a log
write, all sequential).

`run_pipeline_autoloader` uses Databricks Autoloader (`cloudFiles`) instead:
- file discovery is incremental and checkpointed, not a full listing every run
- files are read and written in micro-batches, not one Spark job per file
- logging happens once per micro-batch, not once per file
- no schema extraction — just the raw text, the parsed VARIANT, and file
  metadata, matching the target table columns above

Uses `trigger(availableNow=True)`: it processes everything currently sitting
in the folder as a series of batches and then stops, rather than running as
a continuous stream.

**Three modes, chosen per source via `is_full_load` / `is_incremental_load`:**

| mode | source_control flags | what gets scanned | checkpoint/schema location |
|---|---|---|---|
| Legacy / flat folder | both 0 | `file_path` directly | `tgt_checkpoint_path` |
| Full load | `is_full_load=1` | every `yyyyMMdd` subfolder under `file_path`, every run | `tgt_checkpoint_path_<folder>` per subfolder |
| Incremental load | `is_incremental_load=1` | only today's `yyyyMMdd` subfolder | `tgt_checkpoint_path_<today>` |

Example: `file_path = /Volumes/iics/casemix/history/casemix/zip_files/processed/`
with subfolders `20260901/`, `20260902/`, ...
- Full load scans `.../20260901/`, `.../20260902/`, ... each run, with
  checkpoints at `.../check_point_20260901/checkpoint`,
  `.../check_point_20260902/checkpoint`, etc. A subfolder that's already
  fully loaded is cheap to re-check: Autoloader consults its own
  checkpoint, finds no new files, and the batch completes almost
  instantly - only the one top-level folder listing runs every time.
- Incremental load scans only today's folder (e.g. `.../20260922/` when
  run on 2026-09-22), with checkpoint `.../check_point_20260922/checkpoint`.
  If today's folder doesn't exist yet, the source is skipped for this run
  and a log row records why.

Setting both flags to 1 on the same row is treated as a config error:
the pipeline logs it (status `FAILED`, no file activity) and moves on to
the next source rather than guessing.

In [0]:
def get_checkpoint_path(source_row, folder_name=None):
    base = source_row["tgt_checkpoint_path"].rstrip("/")
    if folder_name:
        base = f"{base}_{folder_name}"
    return f"{base}/checkpoint"


def get_schema_path(source_row, folder_name=None):
    base = source_row["tgt_checkpoint_path"].rstrip("/")
    if folder_name:
        base = f"{base}_{folder_name}"
    return f"{base}/schema"


def _write_autoloader_batch(source_row, tgt_full, scan_path=None):
    def _inner(batch_df, batch_id):
        if batch_df.isEmpty():
            return

        batch_df = batch_df.cache()
        batch_df.write.format("delta").mode("append")\
        .partitionBy("data_year", "data_month")\
        .saveAsTable(tgt_full)

        now = datetime.now()
        log_df = (
            batch_df.select("file_name")
                    .withColumn("log_id", F.expr("uuid()"))
                    .withColumn("run_id", F.lit(run_id))
                    .withColumn("executed_by", F.lit(executed_by))
                    .withColumn("s_no", F.lit(source_row["s_no"]))
                    .withColumn("source", F.lit(source_row["source"]))
                    .withColumn("file_path", F.lit(scan_path or source_row["file_path"]))
                    .withColumn("tgt_catalog", F.lit(source_row["tgt_catalog"]))
                    .withColumn("tgt_schema", F.lit(source_row["tgt_schema"]))
                    .withColumn("tgt_table", F.lit(source_row["tgt_table"]))
                    .withColumn("status", F.lit("SUCCESS"))
                    .withColumn("start_time", F.lit(now))
                    .withColumn("end_time", F.lit(now))
                    .withColumn("rows_processed", F.lit(1).cast(LongType()))
                    .withColumn("message", F.lit("Loaded via Autoloader"))
                    .select(
                        "log_id", "run_id", "executed_by", "s_no", "source", "file_path",
                        "tgt_catalog", "tgt_schema", "tgt_table", "file_name", "status",
                        "start_time", "end_time", "rows_processed", "message",
                    )
        )
        log_df.write.format("delta").mode("append") \
            .saveAsTable(get_full_table_name(meta_catalog, meta_schema, log_table))

        batch_df.unpersist()

    return _inner


def process_source_autoloader(source_row, folder_name=None):

    file_type = source_row["file_type"].lower()
    if file_type != "json":
        raise NotImplementedError(f"file_type '{file_type}' is not supported yet")

    tgt_full = get_full_table_name(source_row["tgt_catalog"], source_row["tgt_schema"], source_row["tgt_table"])

    scan_path = (
        source_row["file_path"] if not folder_name
        else source_row["file_path"].rstrip("/") + "/" + folder_name + "/"
    )

    raw_stream = (
        spark.readStream.format("cloudFiles")
             .option("cloudFiles.format", "text")
             .option("wholetext", "true")
             .option("cloudFiles.schemaLocation", get_schema_path(source_row, folder_name))
             .option("cloudFiles.includeExistingFiles", "true")
             .option("cloudFiles.allowOverwrites", "true")
             .load(scan_path)
    )


    # file-name and file_Date extraction: carereport_hsar_<id>_20220101_20251106_20251113_102927.json -> 20251113
    name_no_ext = F.regexp_extract(F.col("_metadata.file_name"), r"^(.*?)(\.[^.]+)?$", 1)
    dt_data_file_expr = F.element_at(F.split(name_no_ext, "_"), -2)
    file_date_expr = F.to_date(dt_data_file_expr, "yyyyMMdd")

    transformed = raw_stream.select(
        F.lit(scan_path).alias("file_path"),
        F.col("_metadata.file_name").alias("file_name"),
        F.col("value").alias("json_original_data"),
        F.parse_json(F.col("value")).alias("json_data_for_parsing"),
        (F.col("_metadata.file_size") / 1024.0).alias("file_size_kb"),
        F.col("_metadata.file_modification_time").alias("file_modified_date"),
        dt_data_file_expr.alias("dt_data_file"),
        file_date_expr.alias("dt_data_file_date"),
        F.year(file_date_expr).alias("data_year"),
        F.month(file_date_expr).alias("data_month"),
        F.current_timestamp().alias("loaded_at"),
    )

    query = (
        transformed.writeStream
                   .foreachBatch(_write_autoloader_batch(source_row, tgt_full, scan_path))
                   .option("checkpointLocation", get_checkpoint_path(source_row, folder_name))
                   .trigger(availableNow=True)
                   .start()
    )
    query.awaitTermination()
    return query.lastProgress


def process_source_full_load(source_row):
    base_path = source_row["file_path"]

    try:
        subfolders = list_date_subfolders(base_path)
    except Exception as e:
        error_message = f"Full load folder listing failed for {base_path}: {str(e)[:400]}"
        print(f"Source {source_row['s_no']}: {error_message}")
        write_log(status="FAILED", source_row=source_row, message=error_message)
        return

    if not subfolders:
        print(f"Source {source_row['s_no']}: full load - no date subfolders (yyyyMMdd) found under {base_path}")
        return

    print(f"Source {source_row['s_no']}: full load - {len(subfolders)} date folder(s) found: {subfolders}")

    for folder_name in subfolders:
        print(f"Source {source_row['s_no']}: full load - scanning folder {folder_name}")
        try:
            process_source_autoloader(source_row, folder_name=folder_name)
        except Exception as e:
            error_message = f"Full load folder {folder_name} failed: {str(e)[:400]}"
            print(f"Source {source_row['s_no']}: {error_message}")
            write_log(status="FAILED", source_row=source_row, message=error_message)


def process_source_incremental_load(source_row):
    today_folder = datetime.now().strftime("%Y%m%d")
    base_path = source_row["file_path"].rstrip("/")
    folder_path = f"{base_path}/{today_folder}/"

    try:
        dbutils.fs.ls(folder_path)
    except Exception:
        message = f"Incremental load - no folder for today's date ({today_folder}) at {folder_path}, nothing to process"
        print(f"Source {source_row['s_no']}: {message}")
        write_log(status="SUCCESS", source_row=source_row, message=message)
        return

    print(f"Source {source_row['s_no']}: incremental load - scanning today's folder {today_folder}")
    try:
        process_source_autoloader(source_row, folder_name=today_folder)
    except Exception as e:
        error_message = f"Incremental load folder {today_folder} failed: {str(e)[:400]}"
        print(f"Source {source_row['s_no']}: {error_message}")
        write_log(status="FAILED", source_row=source_row, message=error_message)


def run_pipeline_autoloader():

    create_log_table()

    print("Autoloader pipeline run started")
    active_sources = get_active_sources()
    print(f"{len(active_sources)} active source(s) found in source_control")

    for source_row in active_sources:
        is_full = source_row.get("is_full_load") == 1
        is_incremental = source_row.get("is_incremental_load") == 1

        if is_full and is_incremental:
            message = "is_full_load and is_incremental_load are both set to 1 - invalid config, skipping source"
            print(f"Source {source_row['s_no']}: {message}")
            write_log(status="FAILED", source_row=source_row, message=message)
            continue

        print(f"Source {source_row['s_no']}: starting Autoloader ingestion")

        if is_full:
            process_source_full_load(source_row)
        elif is_incremental:
            process_source_incremental_load(source_row)
        else:
            process_source_autoloader(source_row)

        print(f"Source {source_row['s_no']}: Autoloader batch complete")

    print("Autoloader pipeline finished")


## 6. File/Record Versioning

The `create_versioned_table()` function creates a versioned copy of each active target table.

- It retrieves active source configurations using `get_active_sources()`.
- It constructs each target table name using the control-table fields:
  `tgt_catalog`, `tgt_schema`, and `tgt_table`.
- It partitions records using `versioning_partition_field` (for example, `file_name`).
- It assigns a sequential `file_version` for each partition, ordered by `file_modified_date`.
- The versioned data is saved as `<target_table>_versioned` in overwrite mode.

Example:

`iics.casemix.json_test` → `iics.casemix.json_test_versioned`

In [0]:
def create_versioned_table():
    control_rows = get_active_sources()

    for row in control_rows:
        target_table = (
            f"{row['tgt_catalog']}."
            f"{row['tgt_schema']}."
            f"{row['tgt_table']}"
        )
        partition_field = row["versioning_partition_field"]

        df = spark.sql(f"""
            SELECT
                ROW_NUMBER() OVER (
                    PARTITION BY {partition_field}
                    ORDER BY file_modified_date
                ) AS file_version,
                *
            FROM {target_table}
        """)

        (
            df.write
              .mode("overwrite")
              .saveAsTable(f"{target_table}_versioned")
        )

        print(f"Created: {target_table}_versioned")